# Shell Scripting Basics
* `pip` (pip install packages) : `pip` is a package manager just like `apt` but for python libraries, it allows you to install, upgrade and remove libraries.
   * Some Linux distributions prevent pip from modifying Python packages that are managed by the operating system's package manager (such as APT). This helps prevent conflicts and accidental breakage of system tools that depend on those packages.

   **In such cases, it is recommended to use:**
   - Virtual environments (`venv`) for project-specific packages.
   - APT for system-managed Python packages.



* `venv` (Virtual Environment) : An isolated Python environment for a specific project. It allows each project to have its own Python packages and package versions without affecting the operating system or other projects.

   * A virtual environment contains:

   * A Python interpreter
   * pip
   * Installed packages
   * Activation scripts

   Think of it as a private Python installation for a project.

   * Create a virtual environment:

   ```bash
   python -m venv venv
   ```

   * `python` : Runs the Python interpreter.
   * `-m` (module) : Runs a Python module as a program.
   * `venv` : The module used to create virtual environments.
   * The second `venv` is the directory name to create.

   * Activate the virtual environment:

   Linux/macOS:

   ```bash
   source venv/bin/activate
   ```

   Windows:

   ```powershell
   venv\Scripts\activate
   ```

   * After activation, packages installed with `pip` are installed into the virtual environment instead of the system Python installation.

   * Deactivate the virtual environment:

   ```bash
   deactivate
   ```
   * A virtual environment uses the same Python version that created it. The Python executable inside the environment may be a copy or a symbolic link to the system Python executable.

   * A virtual environment only isolates Python-related tools and packages. It does not isolate the shell, operating system, files, networking, or other applications.


* Jupyter Kernel: The process responsible for executing code in notebook cells and returning the results.

  Examples:
  - Python Kernel → Executes Python code.
  - Bash Kernel → Executes Bash commands.
  - R Kernel → Executes R code.

  In Jupyter terminology, a kernel is a code execution engine, not the operating system kernel.

* **Anything you can do in the terminal can generally be automated in a shell script (Bash script).**

In this notebook, we will explore the core concepts of shell scripting. To run the examples interactively in this notebook:
1. Make sure you have the **Bash Kernel** installed. You can install it using pip:
   ```bash
   pip install bash_kernel
   python -m bash_kernel.install
   ```
2. Switch the notebook's kernel to **Bash** from the kernel selection menu in Jupyter (top right corner). Once switched, code cells will execute as standard Bash scripts!

### Debugging and Safety Flags (`set` flags)
When writing production-ready scripts, it is good practice to configure Bash safety settings at the top of the file:
* `set -e` (Exit on Error): Instructs the script to exit immediately if any command returns a non-zero exit status (i.e. fails). This prevents errors from cascading.
* `set -u` (No Unset Variables): Treats referencing an unassigned variable as a fatal error and exits immediately.
* `set -x` (Print Trace): Prints each command to the terminal before executing it (useful for debugging).

In [3]:
# Enable safety flags (uncomment to test)
# set -e
# set -u
set -x

echo "Running under Bash Kernel!"
echo "Current Shell: $SHELL"
echo "Current Date: $(date)" 

: 'note here that $SHELL is an environment variable that holds the path to the current shell being used,
it has an existing value. In this case.
here if we type $date it will output nothing because it is a variable but it doesnt have an existing value.
so we use this command $() which in itself is a command substitution that allows us to execute a command and substitute its output in place.
so we ran 2 commands first one is `date` then we ran$() on it which will substitute the output of the `date` command with the current date and time.
' 

+ echo 'Running under Bash Kernel!'
Running under Bash Kernel!
+ echo 'Current Shell: /bin/bash'
Current Shell: /bin/bash
++ date
+ echo 'Current Date: Mon Jun 22 02:58:22 PM EEST 2026'
Current Date: Mon Jun 22 02:58:22 PM EEST 2026
+ : $'note here that $SHELL is an environment variable that holds the path to the current shell being used,\nit has an existing value. In this case.\nhere if we type $date it will output nothing because it is a variable but it doesnt have an existing value.\nso we use this command $() which in itself is a command substitution that allows us to execute a command and substitute its output in place.\nso we ran 2 commands first one is `date` then we ran$() on it which will substitute the output of the `date` command with the current date and time.\n'


## 1. Variables and Constants

In Bash, you can store data in variables. Here are the core rules:
* **Syntax**: Define variables using `variable_name=value`. 
  > [!IMPORTANT]
  > Do **not** leave spaces around the `=` operator (e.g. `name = "Ahmed"` will result in an error).
* **Data Types**: By default, variables are treated as strings. However, Bash will try to interpret them intelligently if they are numbers or alphanumeric.
* **Integers**: You can explicitly declare an integer variable using `declare -i name=value`.
* **Constants (Readonly)**: You can make a variable read-only (constant) using `declare -r name=value` or `readonly name=value`. Once declared, constants cannot be modified or unset.
* **Persistent Jupyter Sessions & exit 0**:
  Jupyter maintains a persistent shell session. If you run a cell containing `declare -r` a second time, it will throw an error saying the variable is read-only.
  To prevent this and allow re-running cells, we can end the cell with `exit 0`. This terminates and restarts the Bash session for the next execution.

In [4]:
# Define a standard variable (treated as string)
name="Ahmed"
echo "Name: $name"

# Define an integer variable
declare -i age=25
echo "Age: $age"

declare -p name age # Display variable attributes and values

# Define a constant (readonly) variable
declare -r PI=3.14
echo "PI: $PI"

# Attempting to re-assign a constant will fail:
# PI=3.14159

# Reset the Jupyter Bash session so the cell can be re-run without "readonly variable" errors
exit 0

+ name=Ahmed
+ echo 'Name: Ahmed'
Name: Ahmed
+ declare -i age=25
+ echo 'Age: 25'
Age: 25
+ declare -p name age
declare -- name="Ahmed"
declare -i age="25"
+ declare -r PI=3.14
+ echo 'PI: 3.14'
PI: 3.14
+ exit 0
exit
Restarting Bash


## 2. Arrays

An array allows you to store multiple values under a single variable name.

Bash supports two types of arrays:

### 1. Indexed Arrays

- Declared using `declare -a array_name`.
- Elements are identified by numeric indexes starting at `0`.
- Values can be assigned individually or during initialization.
- When initializing an array, elements are separated by spaces, not commas `declare -a names=("Ahmed" "Samy" "Ali")`.
- Access an element using `${array_name[index]}`.
- Access all elements using `${array_name[@]}`.
- Referencing the array without an index (e.g. `$names`) returns the first element (`index 0`).

Example structure:

```text
names
├── [0] → Ahmed
├── [1] → Samy
└── [2] → Ali
```

### 2. Associative Arrays

- Declared using `declare -A array_name`.
- Elements are identified by custom string keys instead of numeric indexes.
- Similar to dictionaries in Python or hash maps in other languages.
- Values can be assigned individually or during initialization `declare -A user=([younis_age]="25" [younis_id]=512 [amr_age]="25")`.
- Access a value using `${array_name[key]}`.

Example structure:

```text
users
├── [younis_age]         → 22
├── [younis_profession]  → Data Engineer
├── [younis_status]      → Single
├── [amr_age]            → 25
├── [amr_profession]     → Developer
└── [amr_status]         → Married
```

Unlike indexed arrays, associative arrays use meaningful names (keys) rather than numeric positions.
> **Note:** Bash arrays store all values as strings. When used in arithmetic expressions, Bash can interpret integer values as numbers, but floating-point values require external tools such as `bc`.

In [5]:
# Indexed Array

declare -a ages=(20 20 20 7)
#or
declare -a ages
ages[0]=20
ages[1]=20
ages[2]=20
ages[3]=7

echo "20+20+20+7 = $(( ages[0] + ages[1] + ages[2] + ages[3] ))"


echo


# Associative Array (Multiple Records)

declare -A users=(
    [younis_age]=22
    [younis_profession]="Data Engineer"
    [younis_status]="Single"
    [amr_age]=25
    [amr_profession]="Developer"
    [amr_status]="Married"
)
declare -A users
users[younis_age]=22
users[younis_profession]="Data Engineer"
users[younis_status]="Single"
users[amr_age]=25
users[amr_profession]="Developer"
users[amr_status]="Married"

echo "Younis:"
echo "  Age: ${users[younis_age]}"
echo "  Profession: ${users[younis_profession]}"
echo "  Status: ${users[younis_status]}"

echo

echo "Amr:"
echo "  Age: ${users[amr_age]}"
echo "  Profession: ${users[amr_profession]}"
echo "  Status: ${users[amr_status]}"

20+20+20+7 = 67

Younis:
  Age: 22
  Profession: Data Engineer
  Status: Single

Amr:
  Age: 25
  Profession: Developer
  Status: Married


## 3. Arithmetic Operations

Bash does not support floating-point arithmetic natively. It has three primary ways of performing math operations:
1. **Double Parentheses**: `(( expression ))` or `$(( expression ))`. This is the most modern, clean, and recommended way to perform integer arithmetic. E.g., `(( result = 2 * 5 ))`.
2. **The `expr` command**: An older POSIX utility. E.g. `expr $num + 5`. Note that spaces are required around operators and variables must be prefixed with `$`.
3. **The `bc` command (Bash Calculator)**: Used for advanced math (such as exponents/powers) and floating-point math. Expressions are passed to `bc` using a pipe (`|`). E.g., `echo "4^2" | bc`.
   * To specify decimal precision, use the `scale` variable inside `bc` (e.g. `scale=2`).

> [!TIP]
> When writing a script, **be consistent**. Pick one math style (ideally double parentheses for integers) and use it throughout to make your code clean and readable.

In [6]:
num=10

# 1. Double Parentheses (Recommended for integer arithmetic)
(( result = num * 5 ))
echo "Integer Multiplication: $result"

# 2. Using the expr command (POSIX style)
# expr is an external command that prints the result to standard output.
# We use command substitution $(...) to capture that output and assign it
# to the variable sum.
sum=$(expr "$num" + 5)
echo "Addition using expr: $sum"

# 3. Using the bc command (For floating-point and advanced math)
# bc reads mathematical expressions from standard input. We use echo to
# send the expression through a pipe and command substitution to capture
# the result.
power=$(echo "4^2" | bc)

# Quotes are not required for "4^2", but they are a good habit when passing
# expressions as text. In the example below, quotes are required because the
# semicolon (;) has special meaning to the shell. The quotes ensure the entire
# expression is passed to bc as text.
division=$(echo "scale=2; 10/3" | bc)

# scale=2 tells bc to keep two digits after the decimal point when
# performing the division.
echo "Power (4^2) using bc: $power"
echo "Floating-point division (10/3) using bc: $division"

# The -l option loads bc's math library and increases the default precision
# (scale=20). It also provides additional mathematical functions such as
# sqrt(), s() (sine), c() (cosine), and l() (natural logarithm). In this
# example, scale=2 provides the desired precision, so -l is not required.

Integer Multiplication: 50
Addition using expr: 15
Power (4^2) using bc: 16
Floating-point division (10/3) using bc: 3.33


## 4. Conditional Statements and Data Validation

### Conditionals
Conditional statements allow a script to make decisions based on conditions.

Bash uses the following structure:

```bash
if condition; then
    # commands
elif another_condition; then
    # commands
else
    # commands
fi
```
if  → start of the conditional block

fi  → end of the conditional block

You can also nest `if` statements inside one another.

### Pattern Matching with `case`
The `case` statement is a cleaner alternative to a long chain of `if-elif` statements when checking a single variable against multiple values or patterns.

```bash
case "$variable" in
    pattern1)
        # commands
        ;;
    pattern2|pattern3)
        # commands
        ;;
    *)
        # default commands (else)
        ;;
esac
```

### Regular Expressions (`=~`)
Within double brackets `[[ ... ]]`, you can perform regular expression checks using the `=~` operator. This is extremely useful for data validation.

Example:

```bash
[[ $input =~ ^[0-9]+$ ]]
```

**Regex Breakdown**
- `^` : Start of the string
- `[0-9]` : Any digit from 0 to 9
- `+` : One or more occurrences
- `$` : End of the string

This pattern ensures that the entire input contains only digits.

### Exit Codes
Every command returns an exit status code when it finishes:

- `0` → Success (True)
- Non-zero → Failure (False)

### Single vs Double Brackets

- `[ ... ]` : Traditional POSIX test syntax
- `[[ ... ]]` : Bash-specific syntax that is safer and supports advanced features such as regular expressions

> [!IMPORTANT]
> Always leave spaces inside brackets:
>
> ```bash
> [[ $a -eq $b ]]
> ```
>
> Incorrect:
>
> ```bash
> [[$a -eq $b]]
> [[ $a -eq $b]]
> ```

### Comparison Operators

#### Numeric Comparisons

- `-eq` : Equal to
- `-ne` : Not equal to
- `-lt` : Less than
- `-le` : Less than or equal to
- `-gt` : Greater than
- `-ge` : Greater than or equal to

#### String Comparisons

- `==` or `=` : Equal to
- `!=` : Not equal to
- `<` or `>` : Lexicographical order (inside `[[ ... ]]`)

In [7]:
# if-elif-else example

score=85

if [[ $score -ge 90 ]]; then
    echo "Grade: A"
elif [[ $score -ge 80 ]]; then
    echo "Grade: B"
else
    echo "Grade: C"
fi

echo

# Example of pattern matching using case

user_role="editor"

case $user_role in
    "admin")
        echo "Welcome Admone!"
        ;;
    "editor")
        echo "welcome editor~"
        ;;
    "viewer")
        echo "Welcome Viewer!"
        ;;
    *)
        echo "Fuuuuucher role maybe?"
        ;;
esac

echo

# Example of data validation using regex

user_input=123

if [[ $user_input =~ ^[0-9]+$ ]]; then
    echo "Success: '$user_input' is a valid integer."

    # Nested conditional statement
    if [[ $(echo "$user_input % 2" | bc) == 0 ]]; then
        echo "The number $user_input is EVEN."
    else
        echo "The number $user_input is ODD."
    fi
else
    echo "Error: '$user_input' is not a valid integer."
fi

echo

# Numeric comparison

a=5
b=10

if [[ $a -lt $b ]]; then
    echo "$a is less than $b"
fi

echo

# String comparison

str1="apple"
str2="orange"

if [[ $str1 != $str2 ]]; then
    echo "The strings are different"
fi

Grade: B

welcome editor~

Success: '123' is a valid integer.
The number 123 is ODD.

5 is less than 10

The strings are different


## 5. The `test` Command and File Checks

### The `test` Command
The `test` command is a highly readable and robust alternative to brackets. The command:
```bash
test 1 -le 5
```
Checks if 1 is less than or equal to 5. It returns an exit code of `0` if true, and `1` if false.

### File Testing Operators
Testing file attributes is crucial in automation. You should always check if a file exists or has correct permissions before performing read/write operations to avoid errors.
Common operators:
* `-e file`: Checks if the file/directory exists.
* `-f file`: Checks if the file exists and is a regular file.
* `-d directory`: Checks if it exists and is a directory.
* `-r file`: Checks if the file is readable.
* `-w file`: Checks if the file is writable.
* `-x file`: Checks if the file is executable.

there are many file types in linux like:

\-  = regular file

d  = directory

l  = symbolic link

In [8]:
# 1. Using the test command
test 5 -le 10
echo "Exit code of test (5 <= 10): $?" # $? prints the exit code of the last command (0 means success/true)

# 2. File Testing
touch sample.txt
chmod +x sample.txt

if [[ -f sample.txt ]]; then
    echo "sample.txt exists and is a regular file."
fi

if [[ -r sample.txt && -x sample.txt ]]; then
    echo "sample.txt is both readable and executable."
fi

# Clean up
rm sample.txt

Exit code of test (5 <= 10): 0
sample.txt exists and is a regular file.
sample.txt is both readable and executable.


## 6. Script Arguments

When executing a script file, you can pass arguments to it:
```bash
./script.sh Ahmed 512 Jan
```
Inside the script, these arguments are accessed using special variables:
* `$0`: The name/path of the script.
* `$1`, `$2`, `$3`...: Positional arguments.
* `$#`: The total number of arguments passed.
* `$*` and `$@`: A list of all arguments.
  * > [!TIP]
    > **Always prefer `$@` over `$*`**. When enclosed in double quotes, `"$@"` preserves each argument as a separate string (e.g. handling arguments containing spaces correctly). `"$*"` joins them all into a single space-separated string.

In [9]:
# Create a temporary script file to demonstrate arguments
cat << 'EOF' > run_demo.sh
#!/bin/bash
echo "Script name (\$0): $0"
echo "Total arguments (\$#): $#"
echo "First argument (\$1): $1"
echo "Second argument (\$2): $2"
echo "All arguments (\$@): $@"
EOF

# Make it executable and run it
chmod +x run_demo.sh
./run_demo.sh Ahmed 512 Jan 25

# Clean up
rm run_demo.sh

Script name ($0): ./run_demo.sh
Total arguments ($#): 4
First argument ($1): Ahmed
Second argument ($2): 512
All arguments ($@): Ahmed 512 Jan 25


## 7. Functions

Functions allow you to reuse code blocks in your scripts.

### Syntax
```bash
my_function() {
    # code here
}
# OR
function my_function {
    # code here
}
```

### Key Rules
* **Invocation**: Invoke a function simply by its name: `my_function` (do **not** write parentheses like `my_function()`):
* **Arguments**: Functions do not have parameters in their definition. You pass arguments to a function just like script arguments (e.g., `my_function arg1 arg2`), and access them inside the function using `$1`, `$2`, etc. These shadow the outer script's arguments.
* **Return Values**:
  * The `return` keyword in Bash only returns an exit status code (0 to 255). It is used to signal success (`0`) or failure (non-zero).
  * To return actual data (like strings or arrays), print the output to standard output (e.g., `echo "data"`) and capture it using command substitution: `result=$(my_function)`.

In [10]:
# Define a function with arguments
greet_user() {
    echo "Hello, $1!"
}

# Call the function
greet_user "Ahmed"

# Function returning exit code
is_even() {
    if (( $1 % 2 == 0 )); then
        return 0 # True/Success
    else
        return 1 # False/Failure
    fi
}

is_even 4
echo "Exit status for 4: $?" # Prints 0 (success)

# Function returning data via stdout
get_date() {
    date +%Y-%m-%d
}

current_date=$(get_date)
echo "Current Date captured from function: $current_date"

Hello, Ahmed!
Exit status for 4: 0
Current Date captured from function: 2026-06-22


## 8. Reading User Input

To make scripts interactive, you can read inputs from standard input using the `read` command:
* **Basic Input**: `read var_name`.
* **Input with Prompt**: Use the `-p` flag to display a prompt message: `read -p "Enter username: " username`.
* **Silent/Secure Input**: Use the `-s` flag to hide the characters typed (ideal for passwords): `read -s -p "Enter password: " password`.

In [11]:
# Prompt for username
#read -p "Enter your username: " username
#echo "Username set to: $username"

# Securely prompt for password
#read -s -p "Enter your password: " password
#echo "" # Print newline since Enter key does not echo it
#echo "Password stored securely."

## 9. Loops

Bash supports three types of loops:
1. **For Loop**:
   * Iterating over a list of items:
     ```bash
     for item in val1 val2 val3; do ... done
     ```
   * C-style for loop:
     ```bash
     for ((i=0; i<10; i++)); do ... done
     ```
   * Iterating over command output:
     ```bash
     for file in $(ls); do ... done
     ```
2. **While Loop**: Executes as long as the condition returns exit status 0 (True).
3. **Until Loop**: Executes as long as the condition returns a non-zero exit status (False), stopping as soon as the condition becomes True.

In [12]:
# 1. For Loop iterating over a list
echo "For Loop (List):"
for name in Ahmed Samy Ali; do
    echo "Hello $name"
done

# 2. C-style For Loop
echo "For Loop (C-Style):"
for ((i=1; i<=3; i++)); do
    echo "Index: $i"
done

# 3. While Loop
echo "While Loop:"
count=1
while [[ $count -le 3 ]]; do
    echo "Count: $count"
    (( count++ ))
done

# 4. Until Loop
echo "Until Loop:"
val=1
until [[ $val -gt 3 ]]; do
    echo "Value: $val"
    (( val++ ))
done

For Loop (List):
Hello Ahmed
Hello Samy
Hello Ali
For Loop (C-Style):
Index: 1
Index: 2
Index: 3
While Loop:
Count: 1
Count: 2
Count: 3
Until Loop:
Value: 1
Value: 2
Value: 3
